In [ ]:
# ==========================================
# 02. PROTOTIPADO E INVESTIGACIÓN DE MODELOS
# ==========================================

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARGA DE DATOS
DATA_PATH = os.path.join("..", "data", "processed", "enemdu_procesado.csv")
df = pd.read_csv(DATA_PATH, sep=';', low_memory=False)

# Definir variable dependiente (binaria) y variables independientes (ejemplo explicativo)
# Sustituye 'empleo_formal' y la lista por tus variables reales
y_col = 'empleo_formal' if 'empleo_formal' in df.columns else df.columns[0]
X_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != y_col][:4]

df_clean = df[[y_col] + X_cols].dropna()

X = df_clean[X_cols]
X_const = sm.add_constant(X)
y = df_clean[y_col]

# 2. PRUEBAS DE MULTICOLINEALIDAD (VIF)
print("\n--- Factor de Inflación de la Varianza (VIF) ---")
vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
print(vif_data)

# 3. ESTIMACIÓN DE MODELOS LOGIT Y PROBIT
print("\n--- Modelo Logit ---")
logit_mod = sm.Logit(y, X_const).fit()
print(logit_mod.summary())

print("\n--- Modelo Probit ---")
probit_mod = sm.Probit(y, X_const).fit()
print(probit_mod.summary())

# 4. EFECTOS MARGINALES PROMEDIO (AME)
print("\n--- Efectos Marginales Promedio (Logit AME) ---")
logit_ame = logit_mod.get_margeff(at='overall', method='dydx')
print(logit_ame.summary())

# 5. MATRIZ DE CONFUSIÓN, SENSIBILIDAD Y ESPECIFICIDAD
y_pred_prob = logit_mod.predict(X_const)
y_pred = (y_pred_prob >= 0.5).astype(int)

cm = confusion_matrix(y, y_pred)
tn, fp, fn, tp = cm.ravel()

sensibilidad = tp / (tp + fn)
especificidad = tn / (tn + fp)

print("\n--- Evaluación de Clasificación ---")
print("Matriz de Confusión:\n", cm)
print(f"Sensibilidad (TPR): {sensibilidad:.4f}")
print(f"Especificidad (TNR): {especificidad:.4f}")

# 6. CURVA ROC Y AUC
fpr, tpr, thresholds = roc_curve(y, y_pred_prob)
auc_val = roc_auc_score(y, y_pred_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='blue', label=f'Curva ROC (AUC = {auc_val:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
plt.title('Curva ROC - Modelo Logit')
plt.legend()
plt.savefig(os.path.join("..", "outputs", "figures", "curva_roc.png"), bbox_inches='tight')
plt.show()